# Splitting-strategy calibration

Systematically sweeps `(backend, dim, AR, sticking, numSplits, splitFactor)` using `sweep.py`, then fits backend-specific predictive models.

The CPU and GPU paths are statistically equivalent in mean flux, but they do **not** have the same throughput optimum. Treat CPU and GPU calibration as separate backends.

**Quality metric**
The sweep selects the depth bin with the fewest raw surface hits per seed. Baseline and split minima are selected independently. The optimization target is `min(hits/seed) / runtime`, so the speedup measures how many actual unweighted ray-surface intersections per seed per second the split strategy produces in its worst-hit bin versus baseline in its own worst-hit bin.

Flux columns are retained only as diagnostics for bias checks; they do not choose the quality bin and do not enter the speedup metric. `N_eff` columns are retained as diagnostics/fallback for old sweep output.

**Workflow**
1. Run CPU and/or GPU sweeps, or reload existing CSVs.
2. Clean data — filter wrong-axis, starved, or pathological runs.
3. For each `(backend, dim, AR, sticking)` cell find the `(numSplits, splitFactor)` pair that maximises worst-hit-bin `hits/seed/s`.
4. Fit a power-law model per backend and dimension.
5. Produce closed-form formulae that can be implemented in the strategy layer.


In [ ]:
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.optimize import curve_fit

mpl.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3})

HERE        = Path(".").resolve()          # examples/trenchSplitting/
BUILD_DIR   = HERE / "../../build"
BINARY      = (BUILD_DIR / "examples/trenchSplitting/trenchSplitting").resolve()
SWEEP_PY    = (HERE / "sweep.py").resolve()

CSV_CPU_2D  = HERE / "sweep_cpu_2d.csv"
CSV_CPU_3D  = HERE / "sweep_cpu_3d.csv"
CSV_GPU_2D  = HERE / "sweep_gpu_2d.csv"
CSV_GPU_3D  = HERE / "sweep_gpu_3d.csv"

# Toggle these depending on what you want to calibrate. GPU supports dim=2 and dim=3.
RUN_CPU     = False
RUN_GPU     = True
DIMS        = [2]          # use [2, 3] for the full calibration

# Use enough seeds for stable N_eff. For quick smoke tests, reduce SEEDS.
RAYS_PER_PT = 30
SEEDS       = 50
PROBE_RAYS  = 20
PROBE_SEEDS = 5

print(f"Binary : {BINARY}  (exists={BINARY.exists()})")
print(f"sweep.py: {SWEEP_PY}  (exists={SWEEP_PY.exists()})")
print(f"RUN_CPU={RUN_CPU} RUN_GPU={RUN_GPU} DIMS={DIMS}")


## 1 — Run the sweep

If both CSVs already exist the cell is a no-op.  
Delete the CSV files to force a re-run.

In [ ]:
CPU_WORKERS = max(1, os.cpu_count() // 2)
CPU_THREADS = 4
GPU_WORKERS = 1      # avoid concurrent processes fighting over one CUDA device
GPU_THREADS = 1

def csv_for(backend: str, dim: int) -> Path:
    return {
        ("cpu", 2): CSV_CPU_2D,
        ("cpu", 3): CSV_CPU_3D,
        ("gpu", 2): CSV_GPU_2D,
        ("gpu", 3): CSV_GPU_3D,
    }[(backend, dim)]

def run_sweep(dim: int, backend: str, out: Path) -> None:
    if out.exists() and out.stat().st_size > 0:
        print(f"{backend} dim={dim}: {out.name} already exists ({out.stat().st_size // 1024} KB) — skipping.")
        return
    use_gpu = backend == "gpu"
    workers = GPU_WORKERS if use_gpu else CPU_WORKERS
    threads = GPU_THREADS if use_gpu else CPU_THREADS
    cmd = [
        sys.executable, str(SWEEP_PY),
        "--binary", str(BINARY),
        "--dim", str(dim),
        "--out", str(out),
        "--workers", str(workers),
        "--threads", str(threads),
        "--rays-per-point", str(RAYS_PER_PT),
        "--seeds", str(SEEDS),
        "--probe-rays", str(PROBE_RAYS),
        "--probe-seeds", str(PROBE_SEEDS),
    ]
    if use_gpu:
        cmd.append("--gpu")
    print("Running:", " ".join(cmd))
    t0 = time.time()
    proc = subprocess.run(cmd, text=True)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"Sweep ({backend}, dim={dim}) failed with exit code {proc.returncode}")
    print(f"{backend} dim={dim} sweep done in {elapsed/60:.1f} min → {out.name}")

for dim in DIMS:
    if RUN_CPU:
        run_sweep(dim, "cpu", csv_for("cpu", dim))
    if RUN_GPU:
        run_sweep(dim, "gpu", csv_for("gpu", dim))


## 2 — Load & clean

In [ ]:
frames = []
for backend, dim, path in [
    ("cpu", 2, CSV_CPU_2D),
    ("cpu", 3, CSV_CPU_3D),
    ("gpu", 2, CSV_GPU_2D),
    ("gpu", 3, CSV_GPU_3D),
]:
    if path.exists() and path.stat().st_size > 0:
        df = pd.read_csv(path)
        df["backend"] = df.get("backend", backend)
        df["dim"] = dim
        frames.append(df)
    else:
        print(f"WARNING: {path.name} not found — skipping {backend} dim={dim}")

if not frames:
    raise RuntimeError("No sweep CSVs found. Enable RUN_CPU/RUN_GPU above or point CSV_* paths to existing files.")

raw = pd.concat(frames, ignore_index=True)
raw["backend"] = raw["backend"].fillna("cpu")
print(f"Raw rows: {len(raw)}")
raw.head(3)


In [ ]:
# Expected depth axis:
#   dim=2 (trench, source POS_Y)  → Y / axis 1
#   dim=3 (cylinder, source POS_Z) → Z / axis 2
EXPECTED_AXIS = {2: "Y", 3: "Z"}
EXPECTED_SPLIT_AXIS = {2: 1, 3: 2}

df = raw.copy()

if "split_axis" in df.columns:
    df["axis_ok"] = df.apply(
        lambda r: int(r["split_axis"]) == EXPECTED_SPLIT_AXIS.get(int(r["dim"]), -1), axis=1
    )
else:
    df["axis_ok"] = df.apply(
        lambda r: r["detected_axis"] == EXPECTED_AXIS.get(int(r["dim"]), ""), axis=1
    )

for col in ["both_starved", "base_starved"]:
    df[col] = df[col].astype(str).str.lower().map(
        {"true": True, "false": False, "1": True, "0": False}
    ).fillna(False)

# Speedup is based on the fewest-hits-per-seed bin, independently selected for baseline
# and split. Keep flux ratios as optional diagnostics only; they are not a
# quality selector and are not part of the validity mask.
df["pathological"] = df["time_split"].fillna(0) > 60.0
if "quality_metric" not in df.columns:
    df["quality_metric"] = "neff"
if "flux_min_ratio" in df.columns:
    df["flux_diagnostic_ratio"] = df["flux_min_ratio"].fillna(1.0)
elif "flux_bot_ratio" in df.columns:
    df["flux_diagnostic_ratio"] = df["flux_bot_ratio"].fillna(1.0)
else:
    df["flux_diagnostic_ratio"] = 1.0

mask_valid = df["axis_ok"] & ~df["both_starved"] & ~df["pathological"]
valid = df[mask_valid].copy()

print(f"Valid rows: {len(valid)} / {len(df)}")
print("Quality metrics:", df["quality_metric"].value_counts(dropna=False).to_dict())
print("Rejected (wrong axis):",  (~df["axis_ok"]).sum())
print("Rejected (both starved):", df["both_starved"].sum())
print("Rejected (pathological):", df["pathological"].sum())
print("Flux diagnostic outside 10%:", (~df["flux_diagnostic_ratio"].between(0.90, 1.10)).sum())

SPEEDUP_CAP = 100.0
valid["speedup"] = valid["speedup"].clip(upper=SPEEDUP_CAP)


## 3 — Exploratory: worst-hit-bin stochastic throughput

Heatmap of **median speedup** over all `(numSplits, splitFactor)` combinations, for each `(AR, sticking)` cell.

Speedup is computed from the minimum recorded per-bin raw hit count per seed divided by runtime. The baseline minimum and split minimum are selected independently. Green means splitting gives more actual surface intersections per seed per second in its worst-hit bin.


In [ ]:
for (backend, dim), sub in valid.groupby(["backend", "dim"]):
    pivot = sub.groupby(["ar", "sticking"])["speedup"].median().unstack("sticking")

    fig, ax = plt.subplots(figsize=(9, 4))
    im = ax.imshow(
        pivot.values, aspect="auto", origin="lower",
        norm=LogNorm(vmin=0.5, vmax=SPEEDUP_CAP),
        cmap="RdYlGn",
    )
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{s:.2f}" for s in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("sticking")
    ax.set_ylabel("AR")
    ax.set_title(f"{backend.upper()} dim={dim} — median speedup (all split params)")
    fig.colorbar(im, ax=ax, label="speedup (log scale)")
    plt.tight_layout()
    plt.show()


## 4 — Best split parameters per cell

For each `(backend, dim, AR, sticking)` find the `(numSplits, splitFactor)` pair with the highest worst-hit-bin `hits/seed/s` speedup.


In [ ]:
idx_best = valid.groupby(["backend", "dim", "ar", "sticking"])["speedup"].idxmax()
best = valid.loc[idx_best].copy().reset_index(drop=True)

print(f"Cells with a best-param row: {len(best)}")
preview_cols = [
    "backend", "dim", "ar", "sticking", "num_splits", "split_factor",
    "quality_metric", "speedup", "hit_min_base", "hit_min_split", "hit_min_ratio",
    "hit_min_base_depth0", "hit_min_base_depth1",
    "hit_min_split_depth0", "hit_min_split_depth1",
    "eff_quality_base", "eff_quality_split", "time_base", "time_split",
    "split_axis", "split_interval", "flux_diagnostic_ratio",
]
best[[c for c in preview_cols if c in best.columns]].head(10)


In [ ]:
for (backend, dim), sub in best.groupby(["backend", "dim"]):
    for col, label in [("num_splits", "optimal numSplits"), ("split_factor", "optimal splitFactor")]:
        pivot = sub.pivot(index="ar", columns="sticking", values=col)

        fig, ax = plt.subplots(figsize=(9, 4))
        vals = pivot.values.astype(float)
        im = ax.imshow(vals, aspect="auto", origin="lower", cmap="viridis")
        vmax = np.nanmax(vals) if np.isfinite(vals).any() else 1
        for i in range(vals.shape[0]):
            for j in range(vals.shape[1]):
                v = vals[i, j]
                if not np.isnan(v):
                    ax.text(j, i, str(int(v)), ha="center", va="center",
                            fontsize=8, color="white" if v > vmax*0.6 else "black")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{s:.2f}" for s in pivot.columns], rotation=45, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_xlabel("sticking")
        ax.set_ylabel("AR")
        ax.set_title(f"{backend.upper()} dim={dim} — {label}")
        fig.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.show()


## 5 — Model fitting

### Physics motivation

For a Lambertian source entering a trench of half-width $w$ and depth $d$ (AR $= d/w$):
- A ray at polar angle $\theta$ (from vertical) bounces every $2w / \tan\theta$ depth units.
- Weight per bounce decays as $(1 - s)$, so weight at depth $z$ scales as
  $\exp(-z \cdot \lambda^{-1})$ where $\lambda$ absorbs the angular average.
- Depth-adaptive splitting with interval $\Delta$ and factor $f$ injects $f$ children
  every $\Delta$ units; to keep N_eff constant we need $f \approx e^{\Delta / \lambda}$.
- Optimal `numSplits` $= d / \Delta = $ AR $/ (\Delta / w)$.

This motivates the ansatz:
$$
n^* = a \cdot \text{AR}^{\alpha} \cdot (-\ln(1-s))^{\beta},
\qquad
f^* = c \cdot (-\ln(1-s))^{\gamma}
$$
with exponents to be fit from the data.

In [ ]:
# Predictor: log(-log(1-s)) and log(AR)
best["log_ar"]  = np.log(best["ar"])
best["neg_log_s"] = -np.log(1 - best["sticking"].clip(upper=0.9999))
best["log_nls"] = np.log(best["neg_log_s"])

# Fit only where splitting gives a meaningful throughput win. Keep this
# backend-specific: GPU often has a smaller win region than CPU.
best_fit = best[best["speedup"] > 1.1].copy()
print(f"Cells used for fitting: {len(best_fit)} / {len(best)}")
best.groupby(["backend", "dim"])["speedup"].describe()


In [ ]:
from scipy.optimize import curve_fit

def power_law_ns(X, log_a, alpha, beta):
    log_ar, log_nls = X
    return log_a + alpha * log_ar + beta * log_nls

def power_law_sf(X, log_c, gamma):
    log_nls, = X
    return log_c + gamma * log_nls

results = {}
for (backend, dim), sub0 in best_fit.groupby(["backend", "dim"]):
    sub = sub0.dropna(subset=["num_splits", "split_factor"])
    if len(sub) < 4:
        print(f"{backend} dim={dim}: too few points ({len(sub)}) — skipping fit")
        continue

    X_ns = (sub["log_ar"].values, sub["log_nls"].values)
    y_ns = np.log(sub["num_splits"].values.astype(float))
    X_sf = (sub["log_nls"].values,)
    y_sf = np.log(sub["split_factor"].values.astype(float))

    popt_ns, _ = curve_fit(power_law_ns, X_ns, y_ns, p0=[0, 1, 0.5], maxfev=5000)
    popt_sf, _ = curve_fit(power_law_sf, X_sf, y_sf, p0=[0.5, 0.3], maxfev=5000)

    a, alpha, beta = np.exp(popt_ns[0]), popt_ns[1], popt_ns[2]
    c, gamma       = np.exp(popt_sf[0]), popt_sf[1]
    results[(backend, dim)] = dict(a=a, alpha=alpha, beta=beta, c=c, gamma=gamma)

    y_ns_pred = power_law_ns(X_ns, *popt_ns)
    y_sf_pred = power_law_sf(X_sf, *popt_sf)
    r2_ns = 1 - np.var(y_ns - y_ns_pred) / np.var(y_ns) if np.var(y_ns) > 0 else np.nan
    r2_sf = 1 - np.var(y_sf - y_sf_pred) / np.var(y_sf) if np.var(y_sf) > 0 else np.nan

    print(f"\n{backend.upper()} dim={dim}  ({len(sub)} cells)")
    print(f"  num_splits*  = {a:.3f} × AR^{alpha:.3f} × (−ln(1−s))^{beta:.3f}   R²={r2_ns:.3f}")
    print(f"  split_factor* = {c:.3f} × (−ln(1−s))^{gamma:.3f}                R²={r2_sf:.3f}")


## 6 — Model validation: predicted vs empirical optimal

In [ ]:
for (backend, dim), p in results.items():
    sub = best_fit[(best_fit["backend"] == backend) & (best_fit["dim"] == dim)].copy()
    nls = sub["neg_log_s"]
    sub["ns_pred"] = np.round(
        p["a"] * sub["ar"]**p["alpha"] * nls**p["beta"]
    ).astype(int).clip(lower=2)
    sub["sf_pred"] = np.round(
        p["c"] * nls**p["gamma"]
    ).astype(int).clip(lower=2)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, col, pred_col, label in [
        (axes[0], "num_splits",  "ns_pred", "numSplits"),
        (axes[1], "split_factor", "sf_pred", "splitFactor"),
    ]:
        ax.scatter(sub[col], sub[pred_col], alpha=0.5, s=25)
        mx = max(sub[col].max(), sub[pred_col].max()) + 1
        ax.plot([1, mx], [1, mx], "k--", lw=1)
        ax.set_xlabel(f"empirical optimal {label}")
        ax.set_ylabel(f"model predicted {label}")
        ax.set_title(f"{backend.upper()} dim={dim} — {label}")
    plt.tight_layout()
    plt.show()


## 7 — Speedup surface: model-optimal parameters vs best-possible

In [ ]:
for (backend, dim), p in results.items():
    sub_all = valid[(valid["backend"] == backend) & (valid["dim"] == dim)].copy()
    nls = -np.log(1 - sub_all["sticking"].clip(upper=0.9999))
    sub_all["ns_model"] = np.round(
        p["a"] * sub_all["ar"]**p["alpha"] * nls**p["beta"]
    ).astype(int).clip(lower=2)
    sub_all["sf_model"] = np.round(
        p["c"] * nls**p["gamma"]
    ).astype(int).clip(lower=2)

    model_match = sub_all[
        (sub_all["num_splits"] == sub_all["ns_model"]) &
        (sub_all["split_factor"] == sub_all["sf_model"])
    ]
    model_pivot  = model_match.groupby(["ar", "sticking"])["speedup"].mean().unstack("sticking")
    best_pivot   = best[(best["backend"] == backend) & (best["dim"] == dim)].pivot(index="ar", columns="sticking", values="speedup")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, pivot, title in [
        (axes[0], best_pivot,  "Best possible speedup"),
        (axes[1], model_pivot, "Speedup with model-predicted params"),
    ]:
        vals = pivot.values.astype(float)
        im = ax.imshow(vals, aspect="auto", origin="lower",
                       norm=LogNorm(vmin=0.5, vmax=SPEEDUP_CAP), cmap="RdYlGn")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{s:.2f}" for s in pivot.columns], rotation=45, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_xlabel("sticking")
        ax.set_ylabel("AR")
        ax.set_title(f"{backend.upper()} dim={dim} — {title}")
        fig.colorbar(im, ax=ax, label="speedup")
    plt.tight_layout()
    plt.show()


## 8 — Where does splitting *not* help?

Low AR and/or low sticking: rays reach the bottom without much attenuation,
so splitting overhead exceeds the gain.

In [ ]:
for (backend, dim), sub in best.groupby(["backend", "dim"]):
    no_gain = sub[sub["speedup"] < 1.05]
    print(f"{backend} dim={dim}: {len(no_gain)}/{len(sub)} cells where split gives <5% gain")
    if len(no_gain):
        print(no_gain[["ar", "sticking", "speedup"]].sort_values("speedup").to_string(index=False))


## 9 — Summary: closed-form formulae for `SplittingStrategy::configure()`

Paste the printed expressions directly into the C++ method.

In [ ]:
print("=" * 65)
print("Recommended auto-parameter formulae")
print("=" * 65)
for (backend, dim), p in results.items():
    geom = "2D trench" if dim == 2 else "3D cylinder"
    print(f"\n--- {backend.upper()} {geom} (dim={dim}) ---")
    print(f"  AR            = depth / featureSize  (detected from geometry)")
    print(f"  nls           = -log(1 - sticking)   (user-supplied)")
    print()
    print(f"  numSplits*    = max(2, round( {p['a']:.4f} * AR^{p['alpha']:.4f} * nls^{p['beta']:.4f} ))")
    print(f"  splitFactor*  = max(2, round( {p['c']:.4f} * nls^{p['gamma']:.4f} ))")
    print()
    print("  C++ snippet:")
    print(f"    const auto nls  = -std::log(1.0 - sticking);")
    print(f"    const int  ns   = std::max(2, (int)std::round( {p['a']:.4f} * std::pow(ar, {p['alpha']:.4f}) * std::pow(nls, {p['beta']:.4f}) ));")
    print(f"    const int  sf   = std::max(2, (int)std::round( {p['c']:.4f} * std::pow(nls, {p['gamma']:.4f}) ));")
print("=" * 65)


## 10 — Sensitivity: how much does speedup degrade with ±1 step around the optimal?

In [ ]:
for (backend, dim), sub_all in valid.groupby(["backend", "dim"]):
    grouped = sub_all.groupby(["ar", "sticking"])
    ratios = []
    for (ar, s), grp in grouped:
        best_row = grp.loc[grp["speedup"].idxmax()]
        ns_opt, sf_opt = best_row["num_splits"], best_row["split_factor"]
        sp_opt = best_row["speedup"]
        for dns in [-1, 0, 1]:
            for dsf in [-1, 0, 1]:
                if dns == 0 and dsf == 0:
                    continue
                nb = grp[
                    (grp["num_splits"] == ns_opt + dns) &
                    (grp["split_factor"] == sf_opt + dsf)
                ]
                if len(nb):
                    ratios.append(sp_opt / nb["speedup"].values[0])

    ratios = np.array(ratios)
    ratios = ratios[np.isfinite(ratios) & (ratios > 0)]
    if len(ratios) == 0:
        continue
    print(f"{backend} dim={dim}: neighbour-step loss — median {np.median(ratios):.2f}x, "
          f"90th-pct {np.percentile(ratios, 90):.2f}x  "
          f"(optimal / neighbour; <1 would mean neighbour is better)")
